**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**


# Báo cáo bài tập về nhà Buổi 12: Tìm kiếm trong môi trường phức tạp

Trong các buổi trước, tôi chủ yếu xét 8-puzzle dưới dạng bài toán tìm kiếm cổ điển: trạng thái ban đầu được biết đầy đủ, trạng thái đích cũng được biết đầy đủ, mỗi hành động tạo ra đúng một trạng thái kế tiếp. Buổi 12 mở rộng giả thiết này sang các môi trường phức tạp hơn. Điểm thay đổi quan trọng là tác nhân không còn luôn biết chính xác mình đang ở đâu, không phải lúc nào cũng biết đầy đủ mục tiêu, và đôi khi một hành động có thể dẫn tới nhiều kết quả khác nhau.

Tôi hiểu cụm “không có start và goal” trong buổi học không nên hiểu theo nghĩa bài toán hoàn toàn vô mục tiêu. Nếu không có bất kỳ mục tiêu nào thì không còn bài toán tìm kiếm theo nghĩa thông thường. Cách hiểu hợp lý hơn là: tác nhân không có **một trạng thái start đơn lẻ và đầy đủ**, hoặc không có **một trạng thái goal đầy đủ** để so sánh trực tiếp như BFS/A*/IDA*. Khi đó, ta phải biểu diễn tri thức của tác nhân bằng **tập các trạng thái có thể xảy ra**, gọi là **belief state**.

## 1. Tìm kiếm khi không biết đầy đủ trạng thái ban đầu

Trong môi trường không quan sát được đầy đủ, tác nhân không biết chính xác trạng thái thật của bài toán. Thay vì lưu một state duy nhất, tôi lưu một tập:

\[
BS = \{s_1, s_2, ..., s_n\}
\]

Mỗi phần tử trong `BS` là một trạng thái có thể là trạng thái thật. Với 8-puzzle, nếu chỉ biết một vài ô, số trạng thái còn lại có thể tăng rất nhanh. Ví dụ, nếu biết trước 4 ô thì còn 5 ô chưa biết, số khả năng là:

\[
5! = 120
\]

Nếu biết ít hơn, kích thước belief state còn lớn hơn nữa. Đây là lý do giảng viên nhắc rằng không thể để belief state quá lớn, vì số lượng trạng thái tăng theo giai thừa.

### Bốn đặc trưng thuật toán

- **Đầy đủ (Completeness)**: có thể đầy đủ nếu belief state hữu hạn, phép sinh trạng thái được kiểm soát và thuật toán duyệt hết được không gian belief state. Trong thực tế, kích thước belief state quá lớn có thể khiến việc duyệt hết không khả thi.
- **Tối ưu (Optimality)**: chỉ tối ưu nếu ta dùng chiến lược tối ưu trên belief state, chẳng hạn BFS/UCS với chi phí phù hợp. Nếu dùng heuristic hoặc giới hạn số mẫu thì thường không còn đảm bảo tối ưu.
- **Thời gian (Time complexity)**: phụ thuộc vào số belief state và số trạng thái bên trong mỗi belief state. Nếu mỗi belief state chứa nhiều trạng thái đơn, chi phí sinh kế tiếp và lọc quan sát đều tăng mạnh.
- **Bộ nhớ (Space complexity)**: lớn hơn tìm kiếm cổ điển vì mỗi nút không chỉ lưu một trạng thái mà lưu cả một tập trạng thái có thể.

## 2. Tìm kiếm với thông tin một phần

Trong trường hợp có một phần thông tin, tôi có thể biết một vài ô của start hoặc goal. Ví dụ, với 8-puzzle, tôi biết các ô chứa `2`, `8`, `3`, và ô trống `0`, còn các ô khác chưa biết. Khi đó, trạng thái ban đầu không phải là một bảng 3x3 đầy đủ mà là một mẫu quan sát:

```text
2 8 3
? ? ?
? 0 ?
```

Tập các trạng thái phù hợp với mẫu này chính là belief state. Sau mỗi hành động và quan sát mới, tôi cập nhật belief state bằng cách loại bỏ những trạng thái không còn phù hợp. Quá trình này được gọi là **lọc belief state**.

Về bản chất, đây là cách chuyển bài toán thiếu thông tin về một bài toán tìm kiếm trên tập trạng thái. Thay vì hỏi “state hiện tại có phải goal không?”, tôi hỏi “mọi trạng thái còn có thể xảy ra có thỏa điều kiện goal đã quan sát không?” hoặc “trạng thái thật được mô phỏng có khớp phần goal đã biết không?”.

## 3. Tìm kiếm trong môi trường hành động không chắc chắn: AND-OR graph

Trong tìm kiếm cổ điển, một hành động dẫn đến đúng một trạng thái kế tiếp. Trong môi trường không chắc chắn, cùng một hành động có thể tạo ra nhiều kết quả. Khi đó, cây tìm kiếm thông thường không còn đủ rõ ràng, vì ta phải phân biệt:

- **OR node**: tác nhân được quyền chọn một hành động.
- **AND node**: môi trường có thể trả về một trong nhiều kết quả của hành động đó, nên kế hoạch phải xử lý được tất cả nhánh kết quả.

Vì vậy, kết quả của AND-OR search không chỉ là một đường đi tuyến tính, mà là một **kế hoạch có điều kiện**. Nghĩa là nếu hành động sinh ra kết quả A thì đi tiếp theo nhánh A, nếu sinh ra kết quả B thì đi tiếp theo nhánh B.

### Bốn đặc trưng thuật toán AND-OR search

- **Đầy đủ**: có thể đầy đủ nếu không gian trạng thái hữu hạn, độ sâu không bị cắt quá sớm và có xử lý vòng lặp.
- **Tối ưu**: bản cơ bản không đảm bảo tối ưu, vì mục tiêu chính là tìm một chính sách giải được mọi kết quả bất định. Muốn tối ưu cần thêm hàm chi phí và so sánh chi phí chính sách.
- **Thời gian**: thường cao hơn tìm kiếm thường vì mỗi hành động phải xét tất cả kết quả có thể xảy ra.
- **Bộ nhớ**: phải lưu cấu trúc kế hoạch phân nhánh, không chỉ lưu một đường đi.

## 4. So sánh với các thuật toán đã học trước

| Nhóm thuật toán | Trạng thái đang xét | Hành động | Kết quả trả về | Điểm khó chính |
|---|---|---|---|---|
| BFS/DFS/UCS/A*/IDA* | Một state đầy đủ | Xác định | Một đường đi | Bùng nổ số nút |
| Belief State Search | Một tập state có thể | Thường xác định | Chuỗi hành động trên belief state | Bùng nổ kích thước `BS` |
| Partial Observation Search | Tập state được lọc bởi quan sát | Xác định hoặc gần xác định | Chuỗi hành động kèm cập nhật quan sát | Lọc và duy trì tri thức |
| AND-OR Search | State hoặc belief state | Không chắc chắn | Kế hoạch có điều kiện | Phải thắng mọi nhánh môi trường |

## 5. Nội dung chương trình minh họa

Trong code cell bên dưới, tôi vẫn dùng bài toán 8-puzzle để giữ tính nhất quán với các buổi trước. Chương trình có ba chế độ:

1. **Sensorless / Belief State**: minh họa trường hợp không biết trạng thái đầy đủ, chỉ thao tác trên tập trạng thái có thể.
2. **Partial Observation**: minh họa trường hợp start và goal chỉ biết một vài ô; sau mỗi bước, belief state được lọc lại.
3. **AND-OR Graph**: minh họa hành động không chắc chắn; mỗi hành động có thể sinh nhiều kết quả và cần hiểu theo cấu trúc OR/AND.

Tôi giữ `PUZZLE_START` và `PUZZLE_GOAL` giống Buổi 11 để các buổi học vẫn đồng nhất về bài toán mẫu.

## 6. Làm rõ sự khác nhau giữa “không biết gì” và “biết một phần”

Sau khi đối chiếu lại, tôi cần tách rõ hai trường hợp này:

- Với **không biết gì / sensorless**, tác nhân không nhìn thấy cả start lẫn goal dưới dạng một bảng đầy đủ. Trong giao diện, tôi biểu diễn start và goal bằng toàn dấu `?`. Belief state lý thuyết khi không biết ô nào là `9! = 362880`. Vì không có quan sát sau hành động, tác nhân không thể xác nhận chắc chắn mình đã đi tới goal; phần trạng thái thật chỉ được dùng như case study ẩn để mô phỏng.
- Với **biết một phần / partial observation**, tác nhân biết trước một vài ô của start và goal. Trong case study này, tôi cho biết các ô `2`, `8`, `3`, và `0`, nên belief state ban đầu là `5! = 120`. Sau mỗi hành động, tác nhân quan sát thêm quân vừa trượt, vì vậy số ô đã biết tăng lên và belief state được lọc nhỏ dần.

Như vậy, hai chế độ không chỉ khác ở số lượng trạng thái trong belief state, mà còn khác ở bản chất tri thức: sensorless không được cập nhật quan sát, còn partial observation có cơ chế lọc belief state sau từng percept.


In [ ]:
# =========================================================================
# BUỔI 12 - TÌM KIẾM TRONG MÔI TRƯỜNG PHỨC TẠP TRÊN 8-PUZZLE
# Sensorless / Belief State, Partial Observation, AND-OR Graph Search
# =========================================================================

import itertools
import random
import tkinter as tk
from tkinter import ttk


PUZZLE_START = ((2, 8, 3), (1, 6, 4), (7, 0, 5))
PUZZLE_GOAL = ((1, 2, 3), (8, 0, 4), (7, 6, 5))

MOVES = [
    (-1, 0, 'Up'),
    (1, 0, 'Down'),
    (0, -1, 'Left'),
    (0, 1, 'Right'),
]

OBSERVED_START_TILES = (2, 8, 3, 0)
OBSERVED_GOAL_TILES = (1, 2, 3, 0)
MAX_BELIEF_SAMPLE = 240
MAX_RENDER_BOARDS = 6


# ===== 8-puzzle helpers =====================================================

def flatten(state):
    return tuple(value for row in state for value in row)


def unflatten(values):
    values = tuple(values)
    return tuple(tuple(values[r * 3 + c] for c in range(3)) for r in range(3))


def find_tile(state, tile):
    for r in range(3):
        for c in range(3):
            if state[r][c] == tile:
                return r, c
    raise ValueError(f'Không tìm thấy ô {tile}')


def find_blank(state):
    return find_tile(state, 0)


def state_to_text(state):
    return ' '.join(str(value) for value in flatten(state))


def pattern_to_text(pattern):
    return ' '.join(str(value) for row in pattern for value in row)


def get_neighbors(state):
    x, y = find_blank(state)
    result = []
    for dx, dy, action in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            board = [list(row) for row in state]
            board[x][y], board[nx][ny] = board[nx][ny], board[x][y]
            result.append((action, tuple(tuple(row) for row in board)))
    return result


def apply_action(state, action):
    for name, neighbor in get_neighbors(state):
        if name == action:
            return neighbor
    return None


def legal_actions_from_blank(blank_pos):
    x, y = blank_pos
    actions = []
    for dx, dy, action in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            actions.append(action)
    return actions


def manhattan_distance(state, goal=PUZZLE_GOAL):
    total = 0
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)
    for r in range(3):
        for c in range(3):
            value = state[r][c]
            if value != 0:
                gr, gc = goal_pos[value]
                total += abs(r - gr) + abs(c - gc)
    return total


def misplaced_tiles(state, goal=PUZZLE_GOAL):
    count = 0
    for r in range(3):
        for c in range(3):
            if state[r][c] != 0 and state[r][c] != goal[r][c]:
                count += 1
    return count


def known_positions_from_tiles(reference_state, tiles):
    return {tile: find_tile(reference_state, tile) for tile in tiles}


def make_pattern(known_positions):
    grid = [['?' for _ in range(3)] for _ in range(3)]
    for tile, (r, c) in known_positions.items():
        grid[r][c] = tile
    return tuple(tuple(row) for row in grid)


def matches_known_positions(state, known_positions):
    return all(state[r][c] == tile for tile, (r, c) in known_positions.items())


def theoretical_belief_size(known_positions):
    return max(1, len(tuple(itertools.permutations(range(9 - len(known_positions))))))


def count_possible_states(known_positions):
    remaining = 9 - len(known_positions)
    total = 1
    for value in range(2, remaining + 1):
        total *= value
    return total


def belief_from_known(known_positions, limit=None):
    fixed = [None] * 9
    used_tiles = set()
    for tile, (r, c) in known_positions.items():
        index = r * 3 + c
        fixed[index] = tile
        used_tiles.add(tile)

    remaining_tiles = [tile for tile in range(9) if tile not in used_tiles]
    remaining_indices = [index for index, value in enumerate(fixed) if value is None]
    belief = []

    for perm in itertools.permutations(remaining_tiles):
        values = list(fixed)
        for index, tile in zip(remaining_indices, perm):
            values[index] = tile
        belief.append(unflatten(values))
        if limit is not None and len(belief) >= limit:
            break

    return belief


def filter_belief_by_known(belief, known_positions):
    return [state for state in belief if matches_known_positions(state, known_positions)]


def moved_tile_for_action(state, action):
    x, y = find_blank(state)
    for dx, dy, name in MOVES:
        if name == action:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                return state[nx][ny], (x, y), (nx, ny)
    return None, None, None


def update_known_after_action(known_positions, true_state, action, reveal_moved_tile=True):
    moved_tile, old_blank, new_blank = moved_tile_for_action(true_state, action)
    if moved_tile is None:
        return dict(known_positions)

    updated = {
        tile: pos
        for tile, pos in known_positions.items()
        if tile not in (0, moved_tile)
    }
    updated[0] = new_blank
    if reveal_moved_tile:
        updated[moved_tile] = old_blank
    return updated


def apply_action_to_belief(belief, action):
    next_states = []
    for state in belief:
        next_state = apply_action(state, action)
        if next_state is not None:
            next_states.append(next_state)
    return list(dict.fromkeys(next_states))


def unknown_pattern():
    return tuple(tuple('?' for _ in range(3)) for _ in range(3))


def belief_union_actions(belief):
    actions = set()
    for state in belief:
        for action, _ in get_neighbors(state):
            actions.add(action)
    return sorted(actions)


def attempt_action_on_belief(belief, action):
    next_states = []
    for state in belief:
        next_state = apply_action(state, action)
        if next_state is None:
            next_state = state
        if next_state not in next_states:
            next_states.append(next_state)
    return next_states


def average_heuristic(states, heuristic_fn):
    if not states:
        return float('inf')
    return sum(heuristic_fn(state) for state in states) / len(states)


def best_action_for_belief(belief, blank_pos, heuristic_fn):
    candidates = []
    for action in legal_actions_from_blank(blank_pos):
        next_belief = apply_action_to_belief(belief, action)
        candidates.append({
            'action': action,
            'next_belief': next_belief,
            'score': average_heuristic(next_belief, heuristic_fn),
        })
    candidates.sort(key=lambda item: item['score'])
    return candidates


def best_sensorless_actions(belief, heuristic_fn):
    candidates = []
    for action in belief_union_actions(belief):
        next_belief = attempt_action_on_belief(belief, action)
        candidates.append({
            'action': action,
            'next_belief': next_belief,
            'score': average_heuristic(next_belief, heuristic_fn),
        })
    candidates.sort(key=lambda item: (item['score'], item['action']))
    return candidates


# ===== Demo generators ======================================================

def sensorless_belief_demo(heuristic_fn=manhattan_distance, max_steps=8):
    """Minh họa no-observation search: agent không nhìn thấy start/goal cụ thể."""
    true_state = PUZZLE_START
    known = {}
    goal_known = {}
    belief = belief_from_known(known, limit=MAX_BELIEF_SAMPLE)
    theoretical_size = count_possible_states(known)
    visible_state = unknown_pattern()

    yield {
        'mode': 'Sensorless / Belief State',
        'step': 0,
        'status': 'Khởi tạo',
        'true_state': true_state,
        'visible_state': visible_state,
        'known': known,
        'goal_known': goal_known,
        'belief': belief,
        'belief_size': len(belief),
        'theoretical_size': theoretical_size,
        'frontier': [],
        'selected_action': None,
        'knowledge_model': 'Không thấy start/goal; không có percept sau hành động',
        'message': 'Sensorless: agent bắt đầu với toàn dấu ?, BS lý thuyết là 9! = 362880; giao diện chỉ lấy mẫu để chạy nhẹ.',
    }

    for step in range(1, max_steps + 1):
        candidates = best_sensorless_actions(belief, heuristic_fn)
        if not candidates:
            yield {
                'mode': 'Sensorless / Belief State',
                'step': step,
                'status': 'Dừng',
                'true_state': true_state,
                'visible_state': visible_state,
                'known': known,
                'goal_known': goal_known,
                'belief': belief,
                'belief_size': len(belief),
                'theoretical_size': theoretical_size,
                'frontier': [],
                'selected_action': None,
                'knowledge_model': 'Không thấy start/goal; không có percept sau hành động',
                'message': 'Không sinh được hành động nào từ belief state mẫu.',
            }
            return

        selected = candidates[0]
        action = selected['action']
        next_true = apply_action(true_state, action)
        if next_true is not None:
            true_state = next_true
        belief = selected['next_belief']
        hidden_goal_reached = true_state == PUZZLE_GOAL

        yield {
            'mode': 'Sensorless / Belief State',
            'step': step,
            'status': 'Goal thật đã đạt (agent không quan sát)' if hidden_goal_reached else 'Đang tìm',
            'true_state': true_state,
            'visible_state': visible_state,
            'known': known,
            'goal_known': goal_known,
            'belief': belief,
            'belief_size': len(belief),
            'theoretical_size': theoretical_size,
            'frontier': [(item['action'], item['score'], item['next_belief'][:3]) for item in candidates],
            'selected_action': action,
            'knowledge_model': 'Không thấy start/goal; không có percept sau hành động',
            'message': f'Sensorless: thử {action}; state nào đi được thì đổi, state nào không hợp lệ thì giữ nguyên; không reveal thêm tile.',
        }
        if hidden_goal_reached:
            return


def partial_observation_demo(heuristic_fn=manhattan_distance, max_steps=12):
    """Minh họa start/goal chỉ biết một vài ô và lọc belief state sau mỗi quan sát."""
    true_state = PUZZLE_START
    known = known_positions_from_tiles(PUZZLE_START, OBSERVED_START_TILES)
    goal_known = known_positions_from_tiles(PUZZLE_GOAL, OBSERVED_GOAL_TILES)
    belief = belief_from_known(known)
    theoretical_size = count_possible_states(known)

    yield {
        'mode': 'Partial Observation',
        'step': 0,
        'status': 'Khởi tạo',
        'true_state': true_state,
        'known': known,
        'goal_known': goal_known,
        'belief': belief,
        'belief_size': len(belief),
        'theoretical_size': theoretical_size,
        'frontier': [],
        'selected_action': None,
        'visible_state': make_pattern(known),
        'knowledge_model': 'Biết một phần start/goal; reveal quân vừa trượt sau mỗi bước',
        'message': 'Partial observation: biết 4 ô nên BS ban đầu có đúng 5! = 120 trạng thái khả dĩ.',
    }

    for step in range(1, max_steps + 1):
        if matches_known_positions(true_state, goal_known):
            yield {
                'mode': 'Partial Observation',
                'step': step,
                'status': 'Đạt goal pattern',
                'true_state': true_state,
                'known': known,
                'goal_known': goal_known,
                'belief': belief,
                'belief_size': len(belief),
                'theoretical_size': len(belief),
                'frontier': [],
                'selected_action': None,
                'visible_state': make_pattern(known),
                'knowledge_model': 'Biết một phần start/goal; reveal quân vừa trượt sau mỗi bước',
                'message': 'Trạng thái thật đã khớp phần goal được quan sát.',
            }
            return

        blank_pos = known[0]
        candidates = best_action_for_belief(belief, blank_pos, heuristic_fn)
        if not candidates:
            return

        selected = candidates[0]
        action = selected['action']
        old_true = true_state
        true_state = apply_action(true_state, action) or true_state
        known = update_known_after_action(known, old_true, action, reveal_moved_tile=True)
        moved_belief = selected['next_belief']
        belief = filter_belief_by_known(moved_belief, known)
        theoretical_size = count_possible_states(known)
        done = matches_known_positions(true_state, goal_known)

        yield {
            'mode': 'Partial Observation',
            'step': step,
            'status': 'Đạt goal pattern' if done else 'Đang tìm',
            'true_state': true_state,
            'known': known,
            'goal_known': goal_known,
            'belief': belief,
            'belief_size': len(belief),
            'theoretical_size': theoretical_size,
            'frontier': [(item['action'], item['score'], item['next_belief'][:3]) for item in candidates],
            'selected_action': action,
            'visible_state': make_pattern(known),
            'knowledge_model': 'Biết một phần start/goal; reveal quân vừa trượt sau mỗi bước',
            'message': f'Partial observation: đi {action}, quan sát quân vừa trượt, sau đó lọc lại belief state.',
        }
        if done:
            return


# ===== AND-OR graph search ==================================================

def nondeterministic_results(state, action):
    intended = apply_action(state, action)
    if intended is None:
        return []

    all_neighbors = dict(get_neighbors(state))
    action_names = [name for name, _ in get_neighbors(state)]
    index = action_names.index(action)
    slip_action = action_names[(index + 1) % len(action_names)]
    slip = all_neighbors[slip_action]

    results = []
    for candidate in (intended, slip):
        if candidate not in results:
            results.append(candidate)
    return results


def and_or_graph_search(start, goal, depth_limit=10):
    """Tìm một conditional plan cơ bản bằng AND-OR search có giới hạn độ sâu."""
    def or_search(state, path, depth):
        if state == goal:
            return {'type': 'goal'}
        if depth == 0 or state in path:
            return None

        actions = sorted(
            get_neighbors(state),
            key=lambda pair: manhattan_distance(pair[1], goal)
        )
        for action, _ in actions:
            outcomes = nondeterministic_results(state, action)
            branches = {}
            failed = False
            for outcome in outcomes:
                subplan = or_search(outcome, path + (state,), depth - 1)
                if subplan is None:
                    failed = True
                    break
                branches[outcome] = subplan
            if not failed:
                return {
                    'type': 'action',
                    'action': action,
                    'branches': branches,
                }
        return None

    return or_search(start, tuple(), depth_limit)


def and_or_demo(max_steps=10):
    true_state = PUZZLE_START
    plan = and_or_graph_search(PUZZLE_START, PUZZLE_GOAL, depth_limit=8)

    yield {
        'mode': 'AND-OR Graph',
        'step': 0,
        'status': 'Khởi tạo',
        'true_state': true_state,
        'known': {tile: find_tile(PUZZLE_START, tile) for tile in range(9)},
        'goal_known': {tile: find_tile(PUZZLE_GOAL, tile) for tile in range(9)},
        'belief': [true_state],
        'belief_size': 1,
        'theoretical_size': 1,
        'frontier': [],
        'selected_action': None,
        'message': 'AND-OR xét OR-node là hành động của agent và AND-node là các kết quả môi trường.',
        'plan_found': plan is not None,
    }

    for step in range(1, max_steps + 1):
        if true_state == PUZZLE_GOAL:
            yield {
                'mode': 'AND-OR Graph',
                'step': step,
                'status': 'Thành công',
                'true_state': true_state,
                'known': {tile: find_tile(true_state, tile) for tile in range(9)},
                'goal_known': {tile: find_tile(PUZZLE_GOAL, tile) for tile in range(9)},
                'belief': [true_state],
                'belief_size': 1,
                'theoretical_size': 1,
                'frontier': [],
                'selected_action': None,
                'message': 'Trạng thái thật đã đạt goal.',
                'plan_found': plan is not None,
            }
            return

        action_rows = []
        for action, _ in get_neighbors(true_state):
            outcomes = nondeterministic_results(true_state, action)
            worst_h = max(manhattan_distance(outcome) for outcome in outcomes)
            action_rows.append((action, worst_h, outcomes))
        action_rows.sort(key=lambda item: item[1])

        selected_action, _, outcomes = action_rows[0]
        true_state = random.choice(outcomes)
        yield {
            'mode': 'AND-OR Graph',
            'step': step,
            'status': 'Đang mô phỏng',
            'true_state': true_state,
            'known': {tile: find_tile(true_state, tile) for tile in range(9)},
            'goal_known': {tile: find_tile(PUZZLE_GOAL, tile) for tile in range(9)},
            'belief': [true_state],
            'belief_size': 1,
            'theoretical_size': 1,
            'frontier': action_rows,
            'selected_action': selected_action,
            'message': f'OR chọn {selected_action}; AND có {len(outcomes)} kết quả, môi trường chọn ngẫu nhiên một kết quả.',
            'plan_found': plan is not None,
        }


# ===== Tkinter GUI ==========================================================

class ComplexSearchApp:
    def __init__(self, root):
        self.root = root
        self.root.title('Buổi 12 - Tìm kiếm trong môi trường phức tạp')
        self.root.geometry('1180x760')
        self.root.minsize(1060, 680)

        self.mode_var = tk.StringVar(value='Partial Observation')
        self.heuristic_var = tk.StringVar(value='Manhattan Distance')
        self.speed_var = tk.IntVar(value=650)
        self.auto_job = None
        self.generator = None
        self.current_frame = None

        self._build_style()
        self._build_layout()
        self.reset()

    def _build_style(self):
        style = ttk.Style()
        try:
            style.theme_use('clam')
        except tk.TclError:
            pass
        style.configure('TFrame', background='#f5f7fb')
        style.configure('Panel.TFrame', background='white', relief='solid', borderwidth=1)
        style.configure('TLabel', background='#f5f7fb', font=('Segoe UI', 10))
        style.configure('Title.TLabel', font=('Segoe UI', 17, 'bold'), foreground='#111827')
        style.configure('Subtitle.TLabel', font=('Segoe UI', 10), foreground='#4b5563')
        style.configure('PanelTitle.TLabel', background='white', font=('Segoe UI', 11, 'bold'))
        style.configure('Metric.TLabel', background='white', font=('Segoe UI', 10))
        style.configure('TButton', font=('Segoe UI', 10))
        style.configure('TCombobox', font=('Segoe UI', 10))

    def _build_layout(self):
        main = ttk.Frame(self.root, padding=16)
        main.pack(fill='both', expand=True)

        header = ttk.Frame(main)
        header.pack(fill='x')
        ttk.Label(header, text='Buổi 12 - Tìm kiếm trong môi trường phức tạp', style='Title.TLabel').pack(anchor='w')
        ttk.Label(
            header,
            text='8-puzzle: belief state, quan sát một phần, và AND-OR graph search',
            style='Subtitle.TLabel'
        ).pack(anchor='w', pady=(2, 10))

        controls = ttk.Frame(main)
        controls.pack(fill='x', pady=(0, 12))

        ttk.Label(controls, text='Chế độ').pack(side='left')
        mode_box = ttk.Combobox(
            controls,
            textvariable=self.mode_var,
            values=['Sensorless / Belief State', 'Partial Observation', 'AND-OR Graph'],
            state='readonly',
            width=26
        )
        mode_box.pack(side='left', padx=(6, 16))
        mode_box.bind('<<ComboboxSelected>>', lambda _event: self.reset())

        ttk.Label(controls, text='Heuristic').pack(side='left')
        heuristic_box = ttk.Combobox(
            controls,
            textvariable=self.heuristic_var,
            values=['Manhattan Distance', 'Misplaced Tiles'],
            state='readonly',
            width=20
        )
        heuristic_box.pack(side='left', padx=(6, 16))
        heuristic_box.bind('<<ComboboxSelected>>', lambda _event: self.reset())

        ttk.Button(controls, text='Reset', command=self.reset).pack(side='left', padx=3)
        ttk.Button(controls, text='Next Step', command=self.next_step).pack(side='left', padx=3)
        ttk.Button(controls, text='Auto Run', command=self.auto_run).pack(side='left', padx=3)
        ttk.Button(controls, text='Stop', command=self.stop_auto).pack(side='left', padx=3)

        ttk.Label(controls, text='Tốc độ').pack(side='left', padx=(18, 4))
        ttk.Scale(controls, from_=150, to=1400, variable=self.speed_var, orient='horizontal', length=140).pack(side='left')

        body = ttk.Frame(main)
        body.pack(fill='both', expand=True)

        left = ttk.Frame(body, style='Panel.TFrame', padding=12)
        left.pack(side='left', fill='both', expand=True, padx=(0, 10))

        center = ttk.Frame(body, style='Panel.TFrame', padding=12)
        center.pack(side='left', fill='both', expand=True, padx=(0, 10))

        right = ttk.Frame(body, style='Panel.TFrame', padding=12)
        right.pack(side='right', fill='both', expand=False)

        ttk.Label(left, text='Trạng thái thật / mẫu đang biết', style='PanelTitle.TLabel').pack(anchor='w')
        self.current_canvas = tk.Canvas(left, width=330, height=330, bg='white', highlightthickness=0)
        self.current_canvas.pack(pady=(10, 4))
        self.current_info = ttk.Label(left, text='', style='Metric.TLabel', wraplength=330)
        self.current_info.pack(anchor='w')

        ttk.Label(center, text='Belief State / Kết quả AND', style='PanelTitle.TLabel').pack(anchor='w')
        self.belief_canvas = tk.Canvas(center, width=430, height=520, bg='white', highlightthickness=0)
        self.belief_canvas.pack(fill='both', expand=True, pady=(10, 0))

        ttk.Label(right, text='Thông tin bước chạy', style='PanelTitle.TLabel').pack(anchor='w')
        self.metrics_text = tk.Text(right, width=38, height=15, font=('Consolas', 10), bg='#ffffff', relief='flat')
        self.metrics_text.pack(fill='x', pady=(10, 10))
        self.metrics_text.configure(state='disabled')

        ttk.Label(right, text='Log', style='PanelTitle.TLabel').pack(anchor='w')
        self.log_text = tk.Text(right, width=38, height=18, font=('Consolas', 9), bg='#111827', fg='#e5e7eb', relief='flat')
        self.log_text.pack(fill='both', expand=True, pady=(10, 0))
        self.log_text.configure(state='disabled')

    def heuristic_fn(self):
        return manhattan_distance if self.heuristic_var.get() == 'Manhattan Distance' else misplaced_tiles

    def make_generator(self):
        mode = self.mode_var.get()
        if mode == 'Sensorless / Belief State':
            return sensorless_belief_demo(self.heuristic_fn())
        if mode == 'Partial Observation':
            return partial_observation_demo(self.heuristic_fn())
        return and_or_demo()

    def reset(self):
        self.stop_auto()
        self.generator = self.make_generator()
        self.current_frame = None
        self.clear_log()
        self.next_step()

    def next_step(self):
        if self.generator is None:
            self.generator = self.make_generator()
        try:
            self.current_frame = next(self.generator)
            self.render(self.current_frame)
            self.append_log(self.current_frame['message'])
            return True
        except StopIteration:
            self.append_log('Thuật toán đã dừng.')
            self.stop_auto()
            return False

    def auto_run(self):
        self.stop_auto()

        def tick():
            if self.next_step():
                self.auto_job = self.root.after(int(self.speed_var.get()), tick)

        tick()

    def stop_auto(self):
        if self.auto_job is not None:
            self.root.after_cancel(self.auto_job)
            self.auto_job = None

    def render(self, frame):
        agent_view = frame.get('visible_state', make_pattern(frame['known']))
        self.draw_board(self.current_canvas, agent_view, 20, 18, 92, 'Agent view')
        info = f"Known pattern: {pattern_to_text(make_pattern(frame['known']))}"
        if frame.get('knowledge_model'):
            info += f"\n{frame['knowledge_model']}"
        self.current_info.configure(text=info)

        self.render_belief(frame)
        self.render_metrics(frame)

    def render_metrics(self, frame):
        frontier_lines = []
        for row in frame.get('frontier', [])[:6]:
            action = row[0]
            score = row[1]
            frontier_lines.append(f'- {action}: score/worst h = {score:.2f}')

        lines = [
            f"Mode      : {frame['mode']}",
            f"Step      : {frame['step']}",
            f"Status    : {frame['status']}",
            f"Action    : {frame.get('selected_action') or '-'}",
            f"BS shown  : {frame['belief_size']}",
            f"BS theory : {frame['theoretical_size']}",
            f"Known     : {len(frame['known'])} tile(s)",
            f"h(hidden): {manhattan_distance(frame['true_state'])}",
            '',
            'Goal pattern:',
            pattern_to_text(make_pattern(frame['goal_known'])),
            '',
            'Frontier / OR candidates:',
            *(frontier_lines or ['- Chưa sinh ứng viên']),
        ]
        if frame['mode'] == 'AND-OR Graph':
            lines.insert(4, f"Plan depth: {'found' if frame.get('plan_found') else 'not found'}")

        self.metrics_text.configure(state='normal')
        self.metrics_text.delete('1.0', 'end')
        self.metrics_text.insert('end', '\n'.join(lines))
        self.metrics_text.configure(state='disabled')

    def render_belief(self, frame):
        canvas = self.belief_canvas
        canvas.delete('all')

        if frame['mode'] == 'AND-OR Graph' and frame.get('frontier'):
            x, y = 14, 24
            canvas.create_text(x, y - 12, text='AND outcomes của các OR action', anchor='w',
                               font=('Segoe UI', 10, 'bold'), fill='#111827')
            for index, row in enumerate(frame['frontier'][:4]):
                action, score, outcomes = row
                canvas.create_text(x, y, text=f"OR {index + 1}: {action}, worst h={score}",
                                   anchor='w', font=('Segoe UI', 9), fill='#374151')
                y += 18
                for outcome in outcomes[:2]:
                    self.draw_board(canvas, outcome, x, y, 38, '')
                    x += 128
                x = 14
                y += 132
            return

        samples = frame['belief'][:MAX_RENDER_BOARDS]
        if not samples:
            canvas.create_text(20, 24, text='Belief state rỗng.', anchor='w', fill='#ef4444')
            return

        canvas.create_text(
            14, 14,
            text=f"Hiển thị {len(samples)} mẫu đầu tiên trong BS",
            anchor='w',
            font=('Segoe UI', 10, 'bold'),
            fill='#111827'
        )
        for index, state in enumerate(samples):
            row = index // 2
            col = index % 2
            x = 14 + col * 205
            y = 42 + row * 155
            self.draw_board(canvas, state, x, y, 42, f'BS #{index + 1}')

    def draw_board(self, canvas, state, x, y, cell, title):
        if title:
            canvas.create_text(x, y - 8, text=title, anchor='w', font=('Segoe UI', 9, 'bold'), fill='#374151')

        for r in range(3):
            for c in range(3):
                value = state[r][c]
                x1 = x + c * cell
                y1 = y + r * cell
                x2 = x1 + cell - 4
                y2 = y1 + cell - 4
                if value == 0:
                    fill = '#dbeafe'
                    text = ''
                elif value == '?':
                    fill = '#f3f4f6'
                    text = '?'
                else:
                    fill = '#ffffff'
                    text = str(value)
                canvas.create_rectangle(x1, y1, x2, y2, fill=fill, outline='#cbd5e1', width=2)
                canvas.create_text(
                    (x1 + x2) / 2,
                    (y1 + y2) / 2,
                    text=text,
                    font=('Segoe UI', max(11, int(cell / 3)), 'bold'),
                    fill='#111827'
                )

    def append_log(self, message):
        self.log_text.configure(state='normal')
        self.log_text.insert('end', message + '\n')
        self.log_text.see('end')
        self.log_text.configure(state='disabled')

    def clear_log(self):
        self.log_text.configure(state='normal')
        self.log_text.delete('1.0', 'end')
        self.log_text.configure(state='disabled')


if __name__ == '__main__':
    root = tk.Tk()
    app = ComplexSearchApp(root)
    root.mainloop()
